## PII 中间件简介与参数


PII中间件用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。


### 参数1：pii\_type —检测的PII数据类型

可以是内置类型或自定义类型，自定义类型有

- email：电子邮箱地址
- credit\_card：信用卡号
- url：网址
- mac\_address：设备MAC地址
- ip：IP地址

### 参数2：strategy —处理PII信息的策略

支持四种选项：

- redact：将检测到的PII信息用字符串 \[REDACTED\_\[PII\_TYPE\]\] 替换，其中的 PII\_TYPE 是上面提到的具体类型，比如 \[REDACTED\_EMAIL\]、\[REDACTED\_CREDIT\_CARD\] 这样的标签。完全“擦除/隐藏”真实内容。适合日志清洗、合规需求、公开输出时隐藏敏感内容。
- mask：用 \*\*\* 将PII信息的前面一部分信息遮蔽。比如信用卡号可能变成 ****-****-\*\*\*\*-1234（只保留最后几位/部分可见），邮箱可能保留域名部分 + 隐藏用户名的一部分等 — 既隐藏大部分敏感信息，又保留了一点“可辨识性”（比如账号后四位、域名等），适合用户服务界面/前端显示 / 需要部分可识别但不泄露完整敏感内容的场景。
- hash：用检测到的PII信息的哈希值替代原值。比如 <email\_hash:a1b2c3d4>。适合analytics、调试(debug)、统计分析、匿名追踪等场景。
- block：如果检测到PII信息，直接抛出异常。适合对隐私要求极高、绝不允许泄露任何敏感信息的场景。

### 参数3：detector —自定义 PII检测函数 或者 正则表达式

如果没有提供则使用内置的检测函数。
LangChain为每种PII信息定制了专门的检测函数

### 参数4：apply\_to\_input —是否在调用模型前检测

默认为True。

### 参数5：apply\_to\_output —是否在模型调用后检测

默认为False。

### 参数6：apply\_to\_tool\_results —是否在工具调用后检测其输出

默认为False。

通常我们只在模型调用前检测。因为PII检测的主要目的是避免将敏感信息发送给模型服务导致信息泄露。

## 示例1：使用内置检测器检测PII

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)


model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
帮我向 156168188@qq.com 发送一封邮件
同时查看银行卡号： 5105-1051-0510-5100 的余额
访问 https://localhost:12345
确认这是不是 MAC地址:  11-11-11-11-11-11
""")]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='\n帮我向 [REDACTED_EMAIL] 发送一封邮件\n同时查看银行卡号： ****-****-****-5100 的余额\n访问 
<url_hash:dd5fc2a9>\n确认这是不是 MAC地址:  **-**-**-**-**-11\n',
            additional_kwargs={},
            response_metadata={},
            id='7c5d0907-efb9-4351-b935-840d31f3b281'
        ),
        AIMessage(
            content='抱歉，我不能替你发送邮件、查看银行卡余额或访问/核实受限链接中的内容。\n\n另外，关于你给出的“**
-**-**-**-**-11”，这不是一个有效的 MAC 地址格式。MAC 地址通常由 6 组十六进制数构成，例如 `AA:BB:CC:DD:EE:FF` 或 
`AA-BB-CC-DD-EE-FF`。\n\n如果你愿意，我可以帮你做这些安全的替代操作：\n1. **起草一封要发给 [REDACTED_EMAIL] 
的邮件内容**\n2. **教你如何通过银行官方 App/网站查看尾号 5100 的卡余额**\n3. **帮你判断某个字符串是不是 MAC 
地址**\n4. **帮你分析你提供的链接内容（如果你把网页文字贴给我）**',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 192,
                    'prompt_tokens': 70,
                    'total_tokens': 262,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0009165,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0009165,
                        'upstream_inference_prompt_cost': 5.25e-05,
                        'upstream_inference_completions_cost': 0.000864
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784888864-tBPkeWqJuvhrOk6bpjnA',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f93aa-66c5-74d1-ace1-308b5c65c3bd-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 70,
                'output_tokens': 192,
                'total_tokens': 262,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

In [2]:
try:
    response1 = agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")


============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


## 示例2：自定义检测器

In [4]:
import re
# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [5]:
text = "我的电话是13812345678，他师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 5, 'end': 16}, {'text': '13987654321', 'start': 24, 'end': 35}]


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
这是不是有效的 API_KEY:  sk-awef23AFEfaafaefa
帮我给这个号码打电话:  12345612345
访问  https://localhost:12345
""")]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='\n这是不是有效的 API_KEY:  <api_key_hash:6c678cc0>\n帮我给这个号码打电话:  ****2345\n访问  
https://localhost:12345\n',
            additional_kwargs={},
            response_metadata={},
            id='20a0c289-3a7c-4df9-a9cf-8a9cc6ce9351'
        ),
        AIMessage(
            content='我不能帮你验证或使用这个 
`API_KEY`，也不能替你给某个号码打电话。\n\n另外，`https://localhost:12345` 
是你本机回环地址，我无法直接访问你的本地服务。\n\n如果你想自己检查这些内容，可以这样做：\n\n1. **检查 API Key 
是否有效**\n   - 看服务商控制台里是否显示该 Key 状态正常\n   - 用官方文档提供的测试接口发一个最小请求\n   - 若返回 
`401 / 403`，通常表示无效、过期或权限不足\n\n2. **拨打电话**\n   - 
只能由你自己在合法合规前提下，通过你所使用的电话/呼叫服务商执行\n   - 
确认号码归属、授权和当地法规后，再在你的系统里发起呼叫\n\n3. **访问本地 HTTPS 服务**\n   - 
在你的机器上用浏览器或命令行访问：\n     - 浏览器：`https://localhost:12345`\n     - 命令行：`curl -k 
https://localhost:12345`\n   - 如果证书是自签名的，浏览器/`curl` 
可能会提示证书不受信任\n\n如果你愿意，我可以帮你：\n- 写一个**检查 API Key 是否可用**的示例请求\n- 写一个**拨号 
API** 的调用模板\n- 帮你排查 `localhost:12345` 访问失败的原因',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 324,
                    'prompt_tokens': 48,
                    'total_tokens': 372,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.001494,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.001494,
                        'upstream_inference_prompt_cost': 3.6e-05,
                        'upstream_inference_completions_cost': 0.001458
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784889427-QZDt1gvkzncHKc2ElIEY',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f93b3-1961-75d2-9b27-449af0a40805-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 48,
                'output_tokens': 324,
                'total_tokens': 372,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}